# 04 — Fine-tuning with LoRA
## LLM Lie Detector | Hallucination Detection Pipeline

This notebook fine-tunes Llama 3.2 3B using LoRA (Low-Rank Adaptation) to classify
whether a given question-answer pair contains a hallucination.

### Goals
- Load and prepare the combined dataset for training
- Configure LoRA for parameter-efficient fine-tuning
- Train the model with experiment tracking via Weights & Biases
- Evaluate against a naive baseline using F1-score, precision, and recall

### Approach
Rather than fine-tuning all 3 billion parameters, we use LoRA to train a small 
adapter — the industry standard approach for efficient LLM fine-tuning.

In [1]:
from dotenv import load_dotenv
import os
import wandb

load_dotenv('../.env')
wandb.login()
print("W&B login successful.")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: itstamimmirza (itstamimmirza-rwth-aachen-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login successful.


In [2]:
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("All imports successful.")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

All imports successful.
CUDA available: True
GPU: NVIDIA GeForce RTX 4080 Laptop GPU


In [3]:
# Load the combined dataset
df = pd.read_csv('../data/combined_dataset.csv')

print(f"Total samples: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts()}")
print(f"\nSample:")
print(df.head(3).to_string())

Total samples: 15918
Label distribution:
label
1    8328
0    7590
Name: count, dtype: int64

Sample:
                                                                                           question                                                 answer  label    source
0  What television adaptation contains a character who was also adapted into a 2005 superhero film?  Batman Begins has a spin-off TV series called Gotham.      1  halueval
1      What punk rock band using horror film imagery was an influence on the Swedish band Entombed?                                                Misfits      0  halueval
2                                           Which lasted longer, Korean War or Battle of Mindanao?                   The Battle of Mindanao lasted longer.      1  halueval


In [4]:
# Split dataset - 90% train, 10% validation
train_df, val_df = train_test_split(
    df, 
    test_size=0.1, 
    random_state=42,
    stratify=df['label']  # ensures balanced split
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"\nTrain label distribution:\n{train_df['label'].value_counts()}")
print(f"\nVal label distribution:\n{val_df['label'].value_counts()}")

Training samples: 14326
Validation samples: 1592

Train label distribution:
label
1    7495
0    6831
Name: count, dtype: int64

Val label distribution:
label
1    833
0    759
Name: count, dtype: int64


In [5]:
from huggingface_hub import login
login()

model_id = "meta-llama/Llama-3.2-3B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Tokenizer loaded.")

Tokenizer loaded.


In [6]:
def format_and_tokenize(row):
    text = f"Question: {row['question']}\nAnswer: {row['answer']}"
    return text

def tokenize_dataset(df, max_length=256):
    texts = [format_and_tokenize(row) for _, row in df.iterrows()]
    
    encodings = tokenizer(
        texts,
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )
    
    dataset = Dataset.from_dict({
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': df['label'].tolist()
    })
    
    return dataset

print("Tokenizing training set...")
train_dataset = tokenize_dataset(train_df)
print("Tokenizing validation set...")
val_dataset = tokenize_dataset(val_df)

print(f"\nTrain dataset: {train_dataset}")
print(f"Val dataset: {val_dataset}")

Tokenizing training set...
Tokenizing validation set...

Train dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 14326
})
Val dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1592
})


In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded.")
print(f"VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.1f} GB")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] LlamaForSequenceClassification LOAD REPORT from: meta-llama/Llama-3.2-3B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.
VRAM used: 6.4 GB


In [8]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,      # sequence classification
    r=16,                             # rank — controls adapter size
    lora_alpha=32,                    # scaling factor
    lora_dropout=0.1,                 # regularization
    target_modules=["q_proj", "v_proj"],  # which layers to adapt
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

W0425 17:54:28.213000 15776 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


trainable params: 4,593,664 || all params: 3,217,349,632 || trainable%: 0.1428


In [9]:
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    return {
        'f1': f1_score(labels, predictions, average='weighted'),
        'precision': precision_score(labels, predictions, average='weighted'),
        'recall': recall_score(labels, predictions, average='weighted'),
    }

In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='../outputs/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='../outputs/logs',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=True,
    report_to="wandb",
    run_name="llama-hallucination-detector-v1"
)

print("Training arguments configured.")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments configured.


In [11]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Trainer ready.")

Trainer ready.
